# Projet 1 — Prévision de consommation électrique

## Prévoir la consommation horaire à horizon 24 h

Ce TP traite une régression multi-horizon : prévoir chacune des 24 prochaines heures à partir de l’historique disponible. On compare les historiques de 24 h, 48 h et 7 jours, les variables calendaires, des baselines, LSTM, TCN et SARIMA.

**Source :** UCI Individual Household Electric Power Consumption, un foyer à Sceaux, décembre 2006 à novembre 2010, mesures à la minute. La cible est Global_active_power, agrégée en puissance moyenne horaire (kW).

**Limites de la source :** aucune température ni météo n’est fournie ; les vacances scolaires non plus. Les jours fériés français sont calculés à partir du calendrier. Les résultats d’un seul foyer ne se généralisent pas automatiquement à un réseau électrique.

## 0 — Environnement et protocole

Le notebook fonctionne depuis le dépôt ou seul dans Google Colab. Il utilise le fichier UCI s'il est déjà présent ; sinon, il télécharge l'archive officielle et l'extrait dans le dossier de travail. Le téléchargement peut prendre quelques minutes. Les modèles avancés sont désactivés au départ pour inspecter d'abord les données et les baselines ; activer les options dans la cellule ci-dessous pour les entraîner.

Découpage chronologique 70/15/15 : entraînement, validation, test. Une origine est évaluée toutes les 24 h pour éviter le chevauchement des fenêtres cibles. MAE et RMSE sont exprimées en kW. MAPE est calculée sur les cibles non nulles et s'interprète avec prudence lorsque la consommation est proche de zéro.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import Request, urlopen
from zipfile import ZipFile
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ImportError:
    sns = None
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
HORIZON = 24
LOOKBACKS = [24, 48, 168]
TRAIN_FRAC, VALID_FRAC = .70, .15
MIN_MINUTES = 45
ORIGIN_STRIDE = 24
RUN_NEURAL = False
RUN_CALENDAR_ABLATION = False
RUN_SARIMA = False

np.random.seed(SEED)
if sns is not None:
    sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

# Colab lance souvent un notebook GitHub depuis /content sans cloner le dépôt.
working_dir = Path.cwd()
candidates = [working_dir, working_dir / "electric-load-forecast", *working_dir.parents]
ROOT = next(
    (p for p in candidates if (p / "requirements.txt").is_file()
     and (p / "notebooks" / "01_TP_prevision_electrique_guide.ipynb").is_file()),
    working_dir,
)
filename = "household_power_consumption.txt"
DATA = ROOT / "data" / "raw" / filename
uploaded_file = working_dir / filename
if uploaded_file.is_file():
    DATA = uploaded_file

if not DATA.is_file():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    url = (
        "https://archive.ics.uci.edu/static/public/235/"
        "individual+household+electric+power+consumption.zip"
    )
    print("Téléchargement des données UCI vers", DATA)
    try:
        with TemporaryDirectory(dir=DATA.parent) as temporary_dir:
            archive_path = Path(temporary_dir) / "uci.zip"
            request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urlopen(request, timeout=120) as response, archive_path.open("wb") as output:
                shutil.copyfileobj(response, output, length=1024 * 1024)
            extracted_path = Path(temporary_dir) / filename
            with ZipFile(archive_path) as archive:
                with archive.open(filename) as source, extracted_path.open("wb") as output:
                    shutil.copyfileobj(source, output, length=1024 * 1024)
            extracted_path.replace(DATA)
    except Exception as exc:
        raise RuntimeError(
            f"Téléchargement UCI impossible. Placez {filename} dans {DATA.parent} "
            "puis relancez cette cellule."
        ) from exc

print("Dossier de travail :", ROOT)
print("Données :", DATA)

## 1 — Comprendre et agréger les mesures

La [fiche UCI du jeu de données](https://archive.ics.uci.edu/dataset/235/individualhouseholdelectricpowerconsumption) décrit **un seul foyer à Sceaux**, observé à la minute de décembre 2006 à novembre 2010. Le fichier contient deux colonnes temporelles et sept mesures numériques :

| Colonne | Sens | Unité source | Agrégation horaire |
|---|---|---:|---|
| Global_active_power | Puissance active totale du foyer ; **cible** | kW | Moyenne |
| Global_reactive_power | Puissance réactive | kW selon UCI | Moyenne |
| Voltage | Tension | V | Moyenne |
| Global_intensity | Intensité | A | Moyenne |
| Sub_metering_1 | Énergie mesurée dans la cuisine | Wh par mesure minute | Somme |
| Sub_metering_2 | Énergie mesurée dans la buanderie | Wh par mesure minute | Somme |
| Sub_metering_3 | Énergie du chauffe-eau et de la climatisation | Wh par mesure minute | Somme |

Une ligne étiquetée **10:00** représente l'intervalle **[09:00, 10:00)** ; elle est connue à 10:00. À cette origine, les 24 cibles vont de 11:00 à 10:00 le lendemain. La cible est une **puissance moyenne horaire en kW**. Pour une heure entièrement observée, l'énergie estimée sur l'heure vaut cette puissance multipliée par 1 h, en kWh.

Le fichier utilise le point-virgule et des dates jour/mois/année. Les valeurs manquantes sont lues comme absentes. Une heure avec moins de 45 mesures valides est exclue de la cible ; les cibles ne sont jamais interpolées.

In [ ]:
FEATURES = [
    "Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity",
    "Sub_metering_1", "Sub_metering_2", "Sub_metering_3",
]
SUBMETERS = ["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
raw = pd.read_csv(DATA, sep=";", na_values=["?", ""], low_memory=False)
raw["timestamp"] = pd.to_datetime(
    raw["Date"].astype(str)+" "+raw["Time"].astype(str),
    format="%d/%m/%Y %H:%M:%S", errors="coerce"
)
for c in FEATURES:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
raw = raw.dropna(subset=["timestamp"]).sort_values("timestamp").set_index("timestamp")
duplicates = int(raw.index.duplicated().sum())
raw = raw.loc[~raw.index.duplicated(keep="first")]
display(pd.DataFrame({
    "valeur":[f"{len(raw):,}",str(raw.index.min()),str(raw.index.max()),"1 minute",duplicates],
}, index=["Mesures","Début","Fin","Fréquence","Doublons retirés"]))
display(pd.DataFrame({
    "type":raw[FEATURES].dtypes.astype(str),
    "manquants":raw[FEATURES].isna().sum(),
    "manquants_%":(100*raw[FEATURES].isna().mean()).round(3),
}, index=FEATURES))

minute = raw[FEATURES]
hourly = minute.resample("h",closed="left",label="right").mean()
counts = minute.resample("h",closed="left",label="right").count()
for c in SUBMETERS:
    hourly[c] = minute[c].resample("h",closed="left",label="right").sum(min_count=1)
hourly, counts = hourly.asfreq("h"), counts.asfreq("h")
for c in FEATURES:
    hourly.loc[counts[c] < MIN_MINUTES,c] = np.nan
target = hourly["Global_active_power"].rename("target_kw")
y, time_index = target.to_numpy(float), hourly.index
n = len(hourly)
train_end, valid_end = int(n*TRAIN_FRAC), int(n*(TRAIN_FRAC+VALID_FRAC))
display(pd.DataFrame({"valeur":[f"{n:,}",str(time_index.min()),str(time_index.max()),
    f"{target.notna().sum():,}",f"{100*target.isna().mean():.2f}%"]},
    index=["Grille horaire","Début","Fin","Cibles valides","Taux cible manquante"]))
print("Borne entraînement :",time_index[train_end],"| borne validation :",time_index[valid_end])
train_target = target.iloc[:train_end].dropna()
descriptive = pd.Series({
    "Moyenne (kW)": train_target.mean(),
    "Médiane (kW)": train_target.median(),
    "Quantile 5 % (kW)": train_target.quantile(.05),
    "Quantile 95 % (kW)": train_target.quantile(.95),
    "Maximum (kW)": train_target.max(),
}, name="Cible horaire — entraînement")
display(descriptive.to_frame().round(3))
print(f"Lecture : {len(raw):,} mesures minute, {n:,} heures sur la grille, "
      f"{target.notna().sum():,} cibles horaires utilisables.")

## 2 — Séparation temporelle

On garde l'ordre des dates : **70 % pour l'entraînement, 15 % pour la validation et 15 % pour le test**. Un tirage aléatoire mélangerait passé et futur. Chaque exemple prédit 24 heures complètes ; ces 24 cibles doivent rester dans une seule partition. Les heures d'historique peuvent provenir de la partition précédente si elles sont déjà connues à l'origine.

Les origines espacées de 24 heures produisent des blocs de cibles sans chevauchement. Tous les modèles sont évalués sur les mêmes origines valides.

In [ ]:
origins_all = np.arange(max(LOOKBACKS)-1,n-HORIZON,ORIGIN_STRIDE)
def complete_origins(origins):
    return np.asarray([i for i in origins if np.isfinite(y[i+1:i+HORIZON+1]).all()],dtype=int)
train_origins = complete_origins(origins_all[origins_all+HORIZON < train_end])
valid_origins = complete_origins(origins_all[(origins_all+1>=train_end)&(origins_all+HORIZON<valid_end)])
test_origins = complete_origins(origins_all[origins_all+1>=valid_end])
trainval_origins = complete_origins(origins_all[origins_all+HORIZON<valid_end])
split_rows = []
for name, start, stop, origins in [
    ("Entraînement", 0, train_end, train_origins),
    ("Validation", train_end, valid_end, valid_origins),
    ("Test", valid_end, n, test_origins),
]:
    split_rows.append({
        "partition": name,
        "début": time_index[start],
        "fin": time_index[stop-1],
        "heures valides": int(target.iloc[start:stop].notna().sum()),
        "origines complètes": len(origins),
        "cibles évaluées": len(origins) * HORIZON,
    })
display(pd.DataFrame(split_rows).set_index("partition"))
assert min(len(train_origins),len(valid_origins),len(test_origins)) > 0

## 3 — Explorer tendance, cycles et corrélations

Les graphiques portent sur l'entraînement pour guider les choix sans consulter le test. La série journalière montre la tendance ; deux semaines horaires donnent les cycles ; la distribution et les profils par heure, jour et mois décrivent la variabilité. Les pointes ne sont pas supprimées automatiquement : elles peuvent être de vrais usages.

Les corrélations entre mesures électriques sont **contemporaines**. Elles servent à comprendre le foyer mais ne prouvent pas qu'une valeur future sera disponible au moment de prévoir. La météo n'est pas présente dans le fichier UCI.

In [ ]:
train = target.iloc[:train_end]
fig,ax = plt.subplots(2,2,figsize=(14,8))
train.resample("D").mean().plot(ax=ax[0,0],title="Moyenne journalière")
train.iloc[:24*14].plot(ax=ax[0,1],title="Deux semaines horaires")
train.dropna().plot.hist(bins=50,ax=ax[1,0],title="Distribution horaire")
train.groupby(train.index.hour).mean().plot(ax=ax[1,1],marker="o",title="Profil moyen par heure")
ax[1,1].set_ylabel("kW")
plt.tight_layout(); plt.show()

weekday = train.groupby(train.index.dayofweek).mean()
weekday.index = ["Lun","Mar","Mer","Jeu","Ven","Sam","Dim"]
display(weekday.rename("Puissance moyenne kW").to_frame().round(3))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
train.groupby(train.index.month).mean().plot(kind="bar", ax=ax[0], title="Puissance moyenne par mois")
ax[0].set_xlabel("Mois"); ax[0].set_ylabel("kW")
(100 * train.isna().resample("MS").mean()).plot(ax=ax[1], title="Heures absentes par mois")
ax[1].set_ylabel("% d'heures")
plt.tight_layout(); plt.show()

daily = train.resample("D").mean().interpolate(method="time", limit=2).dropna()
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
except ImportError:
    print("Décomposition facultative : installer statsmodels pour afficher ce graphique.")
else:
    if len(daily) >= 21:
        seasonal_decompose(daily, model="additive", period=7).plot()
        plt.suptitle("Décomposition hebdomadaire de la moyenne journalière", y=1.02)
        plt.tight_layout(); plt.show()

correlations = hourly.iloc[:train_end][FEATURES].corr()
fig, ax = plt.subplots(figsize=(9, 7))
if sns is not None:
    sns.heatmap(correlations, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
else:
    image = ax.imshow(correlations, cmap="coolwarm", vmin=-1, vmax=1)
    fig.colorbar(image, ax=ax)
    ax.set_xticks(range(len(FEATURES)), FEATURES, rotation=90)
    ax.set_yticks(range(len(FEATURES)), FEATURES)
ax.set_title("Corrélations contemporaines électriques — entraînement")
plt.tight_layout(); plt.show()
print("Météo absente du fichier UCI ; apparier une source horaire avant toute analyse météo.")

## 4 — Features calendaires

Heure, jour de semaine, saison annuelle, week-end et jours fériés sont connus pour les 24 heures futures. Les sinus/cosinus représentent les cycles sans saut artificiel entre la fin et le début d’un cycle. Les vacances scolaires demandent un calendrier externe.

In [ ]:
CAL_COLS = ["hour_sin","hour_cos","weekday_sin","weekday_cos",
            "year_sin","year_cos","weekend","fr_holiday"]
def calendar_features(idx):
    idx = pd.DatetimeIndex(idx)
    h, wd, doy = idx.hour.to_numpy(), idx.dayofweek.to_numpy(), idx.dayofyear.to_numpy()
    days = np.where(idx.is_leap_year,366.,365.)
    out = pd.DataFrame(index=idx)
    out["hour_sin"],out["hour_cos"] = np.sin(2*np.pi*h/24),np.cos(2*np.pi*h/24)
    out["weekday_sin"],out["weekday_cos"] = np.sin(2*np.pi*wd/7),np.cos(2*np.pi*wd/7)
    out["year_sin"],out["year_cos"] = np.sin(2*np.pi*(doy-1)/days),np.cos(2*np.pi*(doy-1)/days)
    out["weekend"] = (wd>=5).astype(float)
    try:
        import holidays
        fr = holidays.country_holidays("FR",years=range(idx.min().year-1,idx.max().year+2))
        out["fr_holiday"] = [float(t.date() in fr) for t in idx]
    except ImportError:
        out["fr_holiday"] = 0.
    return out[CAL_COLS].astype(float)
cal = calendar_features(time_index)
cal_array = cal.to_numpy(np.float32)
display(cal.head())

## 5 — Baseline réelle pour la prévision des 24 prochaines heures

Une baseline est une règle simple dont on mesure les erreurs **sur les données réelles** avant d'entraîner un réseau. À l'origine temporelle i, pour chaque horizon h de 1 à 24 heures :

| Méthode | Prévision de l'heure cible | Idée |
|---|---|---|
| Persistance | Dernière puissance connue à l'origine | Le niveau reste constant pendant 24 h |
| Saisonnier quotidien | Même heure de la veille | Le profil du jour se répète |
| Saisonnier hebdomadaire | Même heure de la semaine précédente | Le profil hebdomadaire se répète |
| Profil moyen 7 jours | Moyenne de cette heure sur les 7 jours précédents | Lisser les jours atypiques |

Ces quatre règles utilisent **uniquement les mesures disponibles à l'origine**. Une valeur historique absente est remplacée par la médiane de la période d'entraînement, et le tableau indique la fréquence de ce remplacement. Les 24 cibles sont identiques pour toutes les méthodes : leur comparaison est donc équitable. On choisit la meilleure méthode avec la MAE de validation ; le test reste réservé à l'évaluation finale.

In [ ]:
def truth(origins):
    return np.stack([y[i+1:i+HORIZON+1] for i in origins])

def score(actual, pred):
    actual, pred = np.asarray(actual), np.asarray(pred)
    assert actual.shape == pred.shape, "Les prédictions et les cibles doivent avoir la même forme."
    a, p = actual.ravel(), pred.ravel()
    ok = np.isfinite(a) & np.isfinite(p)
    a, p = a[ok], p[ok]
    nonzero = np.abs(a) > 1e-6
    return {
        "MAE_kW": mean_absolute_error(a, p),
        "RMSE_kW": np.sqrt(mean_squared_error(a, p)),
        "MAPE_nonzero_pct": (
            100 * np.mean(np.abs((a[nonzero] - p[nonzero]) / a[nonzero]))
            if nonzero.any() else np.nan
        ),
        "n_cibles": len(a),
        "n_MAPE": int(nonzero.sum()),
    }

train_median = float(np.nanmedian(y[:train_end]))
BASELINES = [
    "Persistance", "Saisonnier quotidien",
    "Saisonnier hebdomadaire", "Profil moyen 7 jours",
]

def baseline(name, origins, values=y, fallback=train_median, return_missing=False):
    origins = np.asarray(origins, dtype=int)
    h = np.arange(1, HORIZON + 1)
    if name == "Persistance":
        prediction = np.repeat(values[origins, None], HORIZON, axis=1)
    elif name in ("Saisonnier quotidien", "Saisonnier hebdomadaire"):
        lag = 24 if name == "Saisonnier quotidien" else 168
        positions = origins[:, None] + h[None, :] - lag
        prediction = values[positions].astype(float)
    elif name == "Profil moyen 7 jours":
        days = np.arange(1, 8)
        positions = origins[:, None, None] + h[None, :, None] - 24 * days[None, None, :]
        history = values[positions]
        count = np.isfinite(history).sum(axis=2)
        prediction = np.nansum(history, axis=2) / np.maximum(count, 1)
        prediction[count == 0] = np.nan
    else:
        raise ValueError(f"Baseline inconnue : {name}")
    missing = ~np.isfinite(prediction)
    prediction[missing] = fallback
    prediction = np.maximum(prediction, 0.0)
    if return_missing:
        return prediction, 100 * float(missing.mean())
    return prediction

valid_truth = truth(valid_origins)
val_predictions = {}
rows = []
for name in BASELINES:
    prediction, fallback_pct = baseline(name, valid_origins, return_missing=True)
    val_predictions[name] = prediction
    rows.append({"modèle": name, **score(valid_truth, prediction),
                 "repli_médiane_%": fallback_pct})
baseline_validation = pd.DataFrame(rows).set_index("modèle")
reference_mae = baseline_validation.loc["Persistance", "MAE_kW"]
baseline_validation["gain_vs_persistance_%"] = (
    100 * (1 - baseline_validation["MAE_kW"] / reference_mae)
)
baseline_validation = baseline_validation.sort_values("MAE_kW")
display(baseline_validation.round(3))
best_baseline = baseline_validation.index[0]
print(f"Meilleure baseline sur validation : {best_baseline} "
      f"(MAE = {baseline_validation.loc[best_baseline, 'MAE_kW']:.3f} kW).")

### Comment présenter cette baseline

La **MAE** est l'erreur absolue moyenne en kW sur les 24 heures ; plus elle est faible, mieux c'est. La **RMSE** pénalise davantage les grosses erreurs, importantes lors des pics. Le **MAPE** est affiché sur les cibles non nulles mais peut être très élevé lorsque la consommation est faible. Le taux de **repli médiane** indique combien de valeurs historiques nécessaires étaient manquantes.

Les graphiques suivants rendent la comparaison concrète : erreur par horizon, puis prévision complète d'une journée de validation. La validation sert à choisir la baseline ; aucune observation de test n'est utilisée ici.

In [ ]:
horizon_errors = pd.DataFrame({
    name: np.mean(np.abs(valid_truth - prediction), axis=0)
    for name, prediction in val_predictions.items()
}, index=pd.Index(np.arange(1, HORIZON + 1), name="horizon_h"))
display(horizon_errors.round(3))
horizon_errors.plot(figsize=(11, 4), marker="o")
plt.title("MAE de validation selon l'horizon")
plt.ylabel("MAE (kW)")
plt.xticks(np.arange(1, HORIZON + 1))
plt.tight_layout(); plt.show()

example_row = len(valid_origins) // 2
example_origin = int(valid_origins[example_row])
example_dates = time_index[example_origin+1:example_origin+HORIZON+1]
plt.figure(figsize=(12, 4))
plt.plot(example_dates, valid_truth[example_row], marker="o", label="Mesuré")
plt.plot(example_dates, val_predictions["Saisonnier quotidien"][example_row],
         marker=".", label="Veille")
plt.plot(example_dates, val_predictions["Profil moyen 7 jours"][example_row],
         marker=".", label="Profil moyen 7 jours")
plt.title(f"Une vraie prévision de 24 h, émise le {time_index[example_origin]}")
plt.ylabel("Puissance active moyenne (kW)")
plt.xticks(rotation=45)
plt.legend(); plt.tight_layout(); plt.show()

print(f"À dire à l'oral : {len(raw):,} mesures minute deviennent {n:,} heures, "
      f"dont {target.notna().sum():,} cibles utilisables. "
      f"Sur {len(valid_origins)} origines de validation, soit "
      f"{len(valid_origins)*HORIZON:,} heures prédites, la meilleure baseline est "
      f"{best_baseline}, avec une MAE de "
      f"{baseline_validation.loc[best_baseline, 'MAE_kW']:.3f} kW.")

## 6 — Prétraitement sans fuite et fenêtres multi-horizons

L’imputation et la standardisation sont ajustées sur l’entraînement seulement. Les cibles absentes ne sont pas imputées. Le réseau reçoit l’historique électrique ; l’ablation peut ajouter ou retirer le calendrier historique et futur.

In [ ]:
def preprocess(fit_end):
    imputer=SimpleImputer(strategy="median").fit(hourly[FEATURES].iloc[:fit_end])
    filled=imputer.transform(hourly[FEATURES])
    scaler=StandardScaler().fit(filled[:fit_end])
    xs=scaler.transform(filled).astype(np.float32)
    mean=float(np.nanmean(y[:fit_end]))
    std=float(np.nanstd(y[:fit_end]))
    return xs,mean,(std if std>0 else 1.)
def windows(origins,lookback,xs,mean,std,use_calendar=True):
    xh,xf,yy=[],[],[]
    for i in origins:
        hist=xs[i-lookback+1:i+1]
        if use_calendar:
            hist=np.column_stack([hist,cal_array[i-lookback+1:i+1]])
            future=cal_array[i+1:i+HORIZON+1]
        else:
            future=np.zeros((HORIZON,len(CAL_COLS)),np.float32)
        xh.append(hist); xf.append(future); yy.append((y[i+1:i+HORIZON+1]-mean)/std)
    return tuple(np.asarray(v,np.float32) for v in (xh,xf,yy))
xs_train,mean_train,std_train=preprocess(train_end)

## 7 — Modèles LSTM et TCN

Les deux réseaux prédisent directement les 24 valeurs futures. Le benchmark entraîne chaque architecture avec 24 h, 48 h et 168 h d’historique. Il est désactivé par défaut pour laisser le notebook inspecter les données sans lancer d’entraînement ; passer RUN_NEURAL à True pour obtenir les scores.

In [ ]:
def neural_classes():
    import torch
    from torch import nn
    from torch.utils.data import DataLoader,TensorDataset
    class LSTM(nn.Module):
        def __init__(self,d,f):
            super().__init__(); self.rnn=nn.LSTM(d,32,batch_first=True)
            self.head=nn.Sequential(nn.Linear(32+HORIZON*f,64),nn.ReLU(),nn.Linear(64,HORIZON))
        def forward(self,x,f):
            z,_=self.rnn(x); return self.head(torch.cat([z[:,-1],f.flatten(1)],1))
    class Block(nn.Module):
        def __init__(self,ci,co,dil):
            super().__init__(); pad=2*dil
            self.a=nn.Conv1d(ci,co,3,padding=pad,dilation=dil)
            self.b=nn.Conv1d(co,co,3,padding=pad,dilation=dil)
            self.skip=nn.Conv1d(ci,co,1) if ci!=co else nn.Identity()
        def forward(self,x):
            z=torch.relu(self.a(x)[:,:,:x.shape[-1]])
            z=torch.relu(self.b(z)[:,:,:x.shape[-1]])
            return torch.relu(z+self.skip(x))
    class TCN(nn.Module):
        def __init__(self,d,f):
            super().__init__(); self.conv=nn.Sequential(Block(d,32,1),Block(32,32,2),Block(32,32,4))
            self.head=nn.Sequential(nn.Linear(32+HORIZON*f,64),nn.ReLU(),nn.Linear(64,HORIZON))
        def forward(self,x,f):
            z=self.conv(x.transpose(1,2))[:,:,-1]
            return self.head(torch.cat([z,f.flatten(1)],1))
    return torch,nn,DataLoader,TensorDataset,LSTM,TCN

def fit_net(kind,lookback,origins,xs,mean,std,use_calendar=True,epochs=8):
    torch,nn,Loader,Dataset,LSTM,TCN=neural_classes(); torch.manual_seed(SEED)
    x,f,labels=windows(origins,lookback,xs,mean,std,use_calendar)
    model=(LSTM if kind=="LSTM" else TCN)(x.shape[-1],f.shape[-1])
    ds=Dataset(torch.tensor(x),torch.tensor(f),torch.tensor(labels))
    loader=Loader(ds,batch_size=128,shuffle=True,generator=torch.Generator().manual_seed(SEED))
    opt=torch.optim.Adam(model.parameters(),lr=1e-3); loss_fn=nn.MSELoss()
    for _ in range(epochs):
        for xb,fb,yb in loader:
            opt.zero_grad(); loss=loss_fn(model(xb,fb),yb); loss.backward(); opt.step()
    return model
def predict_net(model,lookback,origins,xs,mean,std,use_calendar=True):
    torch,*_=neural_classes(); x,f,_=windows(origins,lookback,xs,mean,std,use_calendar)
    model.eval()
    with torch.no_grad(): pred=model(torch.tensor(x),torch.tensor(f)).numpy()
    return np.maximum(pred*std+mean,0)

neural_val,neural_models,neural_specs={},{},{}
if RUN_NEURAL:
    for kind in ["LSTM","TCN"]:
        for lookback in LOOKBACKS:
            name=f"{kind} | {lookback} h | calendrier"
            model=fit_net(kind,lookback,train_origins,xs_train,mean_train,std_train)
            neural_models[name]=model
            neural_specs[name]={"kind":kind,"lookback":lookback,"calendar":True}
            neural_val[name]=predict_net(model,lookback,valid_origins,xs_train,mean_train,std_train)
    display(pd.DataFrame({k:score(valid_truth,v) for k,v in neural_val.items()}).T.sort_values("MAE_kW").round(4))
else:
    print("Réseaux désactivés : activer RUN_NEURAL et réexécuter cette cellule.")

## 8 — SARIMA et ablation calendrier (options)

SARIMA est ajusté sur les 120 derniers jours disponibles avec une saisonnalité quotidienne, prévoit 24 h à chaque origine, puis met à jour son état avec les observations mesurées. Les paramètres restent fixes pendant validation. L’ablation compare la meilleure configuration neuronale avec et sans calendrier.

In [ ]:
def sarima_predict(fit_end,origins,fit_days=120):
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    start=max(0,fit_end-fit_days*24)
    series=pd.Series(y[start:fit_end],index=time_index[start:fit_end])
    fitted=SARIMAX(series,order=(1,0,1),seasonal_order=(1,0,1,24),
        enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=40)
    last=fit_end-1; output=[]
    for origin in origins:
        if origin>last:
            upd=pd.Series(y[last+1:origin+1],index=time_index[last+1:origin+1])
            fitted=fitted.append(upd,refit=False); last=origin
        output.append(np.maximum(np.asarray(fitted.get_forecast(HORIZON).predicted_mean),0))
        stop=origin+HORIZON+1
        upd=pd.Series(y[origin+1:stop],index=time_index[origin+1:stop])
        fitted=fitted.append(upd,refit=False); last=stop-1
    return np.stack(output)

if RUN_SARIMA:
    sarima_val=sarima_predict(train_end,valid_origins)
    val_predictions["SARIMA"]=sarima_val
else:
    sarima_val=None
    print("SARIMA désactivé : activer RUN_SARIMA pour lancer l’ajustement.")

if RUN_CALENDAR_ABLATION and neural_specs:
    best=min(neural_val,key=lambda k:score(valid_truth,neural_val[k])["MAE_kW"])
    spec=neural_specs[best]; rows=[]
    for use_cal in [False,True]:
        model=fit_net(spec["kind"],spec["lookback"],train_origins,xs_train,mean_train,std_train,use_calendar=use_cal)
        pred=predict_net(model,spec["lookback"],valid_origins,xs_train,mean_train,std_train,use_calendar=use_cal)
        rows.append({"calendrier":use_cal,**score(valid_truth,pred)})
    display(pd.DataFrame(rows).set_index("calendrier").round(4))
elif RUN_CALENDAR_ABLATION:
    print("Activer d’abord le benchmark LSTM/TCN.")

## 9 — Comparer sur validation et analyser les erreurs

Le modèle est choisi sur validation selon MAE sur tous les horizons. RMSE et MAPE restent complémentaires. Les pics sont définis par le 95e percentile de l’entraînement. Ne pas modifier les choix après avoir consulté le test.

In [ ]:
all_val=dict(val_predictions); all_val.update(neural_val)
validation_table=pd.DataFrame({k:score(valid_truth,v) for k,v in all_val.items()}).T.sort_values("MAE_kW")
display(validation_table.round(4))
selected=str(validation_table.index[0])
print("Choix validation :",selected)
selected_val=all_val[selected]
h_mae=np.mean(np.abs(valid_truth-selected_val),axis=0)
pd.Series(h_mae,index=np.arange(1,25),name="MAE kW").plot(marker="o",figsize=(10,4))
plt.title("MAE de validation par horizon"); plt.xlabel("Horizon (heures)"); plt.xticks(np.arange(1,25))
plt.tight_layout(); plt.show()
threshold=float(np.nanquantile(y[:train_end],.95))
peaks=valid_truth>=threshold
display(pd.DataFrame([
    {"segment":"Toutes",**score(valid_truth,selected_val)},
    {"segment":"Pics",**score(valid_truth[peaks],selected_val[peaks])}
]).set_index("segment").round(4))

## 10 — Test final, intervalles et compte rendu

Les choix d’architecture et d’historique sont figés à partir de la validation. Pour le test, les transformations et les poids sont réajustés avec les données disponibles jusqu’à sa borne. Les intervalles sont des quantiles absolus des résidus de validation par horizon ; ils fournissent une couverture empirique, pas une garantie.

In [ ]:
test_truth = truth(test_origins)
test_fallback = float(np.nanmedian(y[:valid_end]))
baseline_test_predictions = {}
baseline_test_rows = []
for name in BASELINES:
    prediction, fallback_pct = baseline(
        name, test_origins, fallback=test_fallback, return_missing=True
    )
    baseline_test_predictions[name] = prediction
    baseline_test_rows.append({
        "modèle": name, **score(test_truth, prediction),
        "repli_médiane_%": fallback_pct,
    })
print("Baselines déclarées avant le test, évaluées sur les mêmes cibles :")
display(pd.DataFrame(baseline_test_rows).set_index("modèle").sort_values("MAE_kW").round(3))
if selected in BASELINES:
    test_pred = baseline_test_predictions[selected]
elif selected.startswith(("LSTM","TCN")):
    spec=neural_specs[selected]
    xs_tv,mean_tv,std_tv=preprocess(valid_end)
    model=fit_net(spec["kind"],spec["lookback"],trainval_origins,xs_tv,mean_tv,std_tv)
    test_pred=predict_net(model,spec["lookback"],test_origins,xs_tv,mean_tv,std_tv)
elif selected=="SARIMA":
    test_pred=sarima_predict(valid_end,test_origins)

display(pd.DataFrame([score(test_truth,test_pred)],index=[selected]).round(4))
q90=np.quantile(np.abs(valid_truth-selected_val),.90,axis=0)
i=int(test_origins[0]); dates=time_index[i+1:i+HORIZON+1]
lo=np.maximum(0,test_pred[0]-q90); hi=test_pred[0]+q90
plt.figure(figsize=(12,5))
plt.plot(dates,test_truth[0],marker="o",label="Observé")
plt.plot(dates,test_pred[0],marker="o",label="Prévu")
plt.fill_between(dates,lo,hi,alpha=.2,label="Intervalle empirique 90 %")
plt.title(f"Prévision 24 h — origine {time_index[i]}")
plt.ylabel("Puissance moyenne (kW)"); plt.xticks(rotation=45); plt.legend()
plt.tight_layout(); plt.show()
coverage=np.mean((test_truth>=np.maximum(0,test_pred-q90))&(test_truth<=test_pred+q90))
print(f"Couverture empirique sur le test : {coverage:.1%}")

## Compte rendu et références

### Trame courte pour la présentation

1. **Jeu de données :** un foyer, 2 colonnes temporelles et 7 mesures électriques à la minute. Expliquer kW pour la cible et Wh pour les sous-compteurs.
2. **Préparation :** agréger à l'heure, exclure les heures avec moins de 45 minutes valides, montrer la distribution et les cycles journaliers/hebdomadaires.
3. **Évaluation :** découpage temporel, 24 prévisions par origine, mêmes cibles pour tous les modèles, MAE principale et RMSE pour les pics.
4. **Baseline :** comparer persistance, veille, semaine précédente et profil moyen de 7 jours ; montrer le tableau et une journée de prévisions.
5. **Suite :** présenter LSTM/TCN/SARIMA seulement après avoir établi ce niveau de référence, puis discuter les erreurs et les limites.

Compléter avec les résultats effectivement calculés, sans inventer de score.

- Cible, heure d’émission et horizon 24 h : …
- Période, taux de données manquantes et fréquence après agrégation : …
- Baseline de référence et MAE : …
- Résultats LSTM / TCN / SARIMA et historique retenu : …
- Impact du calendrier : …
- Horizon le plus difficile, erreur sur les pics et couverture empirique : …
- Limites : un foyer, période 2006–2010, météo et vacances scolaires absentes.

Pistes : apparier météo horaire ; tester des validations glissantes ; étudier N-BEATS ou LSTNet ; comparer plusieurs foyers ; évaluer le coût de sous-prévision et de sur-prévision.

- Lai et al. (2018), LSTNet : https://arxiv.org/abs/1703.07015
- Oreshkin et al. (2020), N-BEATS : https://arxiv.org/abs/1905.10437
- UCI Household Electric Power Consumption : https://archive.ics.uci.edu/ml/datasets/individual+household+electric+power+consumption